실험 결과 시각화 코드

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

# 1. 한글 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 2. 초기값 설정 (평가지표 기준)
metrics = ['Accuracy (정확도)', 'Precision (정밀도)', 'Recall (재현율)', 'F1-Score']

# 각 모델별 지표를 담을 리스트: [Accuracy, Precision, Recall, F1] 순서
bert_base = [0.0, 0.0, 0.0, 0.0]
roberta_base = [0.0, 0.0, 0.0, 0.0]
bert_prop = [0.0, 0.0, 0.0, 0.0]
roberta_prop = [0.0, 0.0, 0.0, 0.0]

# 3. 자동 경로 설정 및 metrics 폴더 생성
current_dir = os.path.dirname(os.path.abspath(__file__))
parent_dir = os.path.dirname(current_dir)
result_dir = os.path.join(parent_dir, 'result')
metrics_dir = os.path.join(result_dir, 'metrics')

os.makedirs(metrics_dir, exist_ok=True)

csv_pattern = os.path.join(result_dir, '*.csv')
csv_files = glob.glob(csv_pattern)

print("=== 전체 평가지표 데이터 탐색 시작 ===")
if len(csv_files) == 0:
    print(f"경고: '{result_dir}' 경로에 읽을 수 있는 CSV 파일이 없습니다.")
else:
    for file_path in csv_files:
        print(f"파일 읽기 시도: {file_path}")
        try:
            df = pd.read_csv(file_path)
            df.columns = df.columns.str.strip()
            
            # 4가지 평가지표 컬럼이 모두 존재하는지 확인
            required_cols = ['model', 'accuracy', 'precision', 'recall', 'f1']
            if all(col in df.columns for col in required_cols):
                for index, row in df.iterrows():
                    model_name = str(row['model']).strip()
                    
                    try:
                        acc = float(row['accuracy'])
                        pre = float(row['precision'])
                        rec = float(row['recall'])
                        f1 = float(row['f1'])
                    except ValueError:
                        continue
                    
                    # 지표 리스트 생성
                    metrics_values = [acc, pre, rec, f1]
                    print(f" -> 데이터 인식 완료: [{model_name}] Acc:{acc:.4f}, Pre:{pre:.4f}, Rec:{rec:.4f}, F1:{f1:.4f}")

                    # 모델명 키워드 매칭을 통한 배열 할당
                    if 'BERT_comment_only' in model_name:
                        bert_base = metrics_values
                    elif 'RoBERTa_comment_only' in model_name:
                        roberta_base = metrics_values
                    elif 'BERT_context+comment' in model_name:
                        bert_prop = metrics_values
                    elif 'RoBERTa_context' in model_name:
                        roberta_prop = metrics_values
            else:
                print(f" -> 경고: '{file_path}' 파일에 필수 평가지표 컬럼이 부족합니다.")
                    
        except Exception as e:
            print(f" -> 파일 읽기 오류 ({file_path}): {e}")

print("=== 데이터 병합 완료 ===")

# 4. 그래프 레이아웃 설정
x = np.arange(len(metrics))
width = 0.2  # 한 지표당 4개의 막대가 들어가야 하므로 막대 너비를 0.2로 좁게 설정

# 16개의 막대가 들어가므로 캔버스 가로 길이를 여유 있게 설정
fig, ax = plt.subplots(figsize=(14, 7))

# 5. 막대 그래프 생성 (회색톤: 베이스라인 / 파란색톤: 제안모델)
rects1 = ax.bar(x - 1.5*width, bert_base, width, label='BERT 베이스라인', color='#D3D3D3')
rects2 = ax.bar(x - 0.5*width, roberta_base, width, label='RoBERTa 베이스라인', color='#A9A9A9')
rects3 = ax.bar(x + 0.5*width, bert_prop, width, label='BERT 제안 모델', color='#87CEFA')
rects4 = ax.bar(x + 1.5*width, roberta_prop, width, label='RoBERTa 제안 모델', color='#4169E1')

# 6. 축, 제목, 범례 설정
ax.set_ylabel('Score', fontsize=12)
ax.set_title('입력 방식에 따른 모델별 종합 평가지표 비교', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)

# 수치 텍스트가 잘리지 않도록 y축 최댓값을 1.15로 여유 있게 설정
ax.set_ylim(0, 1.15) 

# 범례를 위쪽 중앙에 가로 4열로 넓게 배치
ax.legend(fontsize=11, loc='upper center', ncol=4)

# 7. 막대 상단 수치 표시 함수
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        if height > 0:
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9, fontweight='bold')

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)
autolabel(rects4)

plt.tight_layout()

# 8. 고화질 이미지 파일을 result/metrics 폴더에 저장 (새로운 이름으로 저장)
output_filename = os.path.join(metrics_dir, 'all_metrics_comparison.png')
plt.savefig(output_filename, dpi=300)
print(f"성공: 종합 지표 그래프가 '{output_filename}' 경로에 저장되었습니다.")